In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
import random

# ==========================================
# 1. WCZYTANIE I PRZYGOTOWANIE DANYCH (Twoja konfiguracja)
# ==========================================

filename = 'jena_climate_10k.csv'


df = pd.read_csv(filename)

# --- KONFIGURACJA KOLUMN ---
TARGET_COL = 'T (degC)'

# Lista kolumn do usunięcia wskazana przez Ciebie
COLS_TO_DROP = [
    'Date Time',
    'Tpot (K)',
    'VPact (mbar)',
    'H2OC (mmol/mol)',
    'wv (m/s)',
    'VPmax (mbar)',
    'Tdew (degC)'
]

# Dodajemy 'Set' do usuwanych, jeśli istnieje w pliku (na wszelki wypadek)
if 'Set' in df.columns:
    COLS_TO_DROP.append('Set')

# Ostateczna lista cech (wszystko co nie jest celem i nie jest do usunięcia)
FEATURE_COLS = [c for c in df.columns if c not in COLS_TO_DROP and c != TARGET_COL]

print(f"Liczba cech wejściowych: {len(FEATURE_COLS)}")
print(f"Cechy: {FEATURE_COLS}")

# --- PODZIAŁ DANYCH (70% / 15% / 15%) ---
n = len(df)
train_split = int(n * 0.70)
val_split   = int(n * 0.85)  # 70% + 15% = 85%

train_df = df[0 : train_split]
val_df   = df[train_split : val_split]
test_df  = df[val_split :]

print(f"Podział danych: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")


# ==========================================
# 2. SKALOWANIE I TWORZENIE SEKWENCJI
# ==========================================

# --- SKALOWANIE ---
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit tylko na zbiorze treningowym, aby uniknąć wycieku danych
X_train_raw = scaler_X.fit_transform(train_df[FEATURE_COLS].values)
y_train_raw = scaler_y.fit_transform(train_df[[TARGET_COL]].values)

# Transformacja walidacji i testu na podstawie scalerów z treningu
X_val_raw = scaler_X.transform(val_df[FEATURE_COLS].values)
y_val_raw = scaler_y.transform(val_df[[TARGET_COL]].values)

X_test_raw = scaler_X.transform(test_df[FEATURE_COLS].values)
# Do ostatecznej weryfikacji używamy oryginalnych wartości targetu (niezeskalowanych)
y_test_raw_orig = test_df[[TARGET_COL]].values

# Funkcja tworząca sekwencje (windowing)
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

Liczba cech wejściowych: 7
Cechy: ['p (mbar)', 'rh (%)', 'VPdef (mbar)', 'sh (g/kg)', 'rho (g/m**3)', 'max. wv (m/s)', 'wd (deg)']
Podział danych: Train=7000, Val=1500, Test=1500


In [ ]:
# ==========================================
# 3. BUDOWNICZY MODELU (LSTM)
# ==========================================
def build_lstm_model(input_shape, params):
    model = Sequential()
    model.add(Input(shape=input_shape))

    # Warstwy LSTM
    for i in range(params['num_layers']):
        # return_sequences=True musi być dla wszystkich warstw POZA ostatnią warstwą LSTM
        return_seq = True if i < params['num_layers'] - 1 else False

        model.add(LSTM(units=params['units'],
                       return_sequences=return_seq,
                       activation='tanh'))

        # Dropout
        model.add(Dropout(0.2))

    # Warstwy Dense
    model.add(Dense(params['dense_neurons'], activation='relu'))
    model.add(Dense(1))

    # Optymalizator
    lr = params['learning_rate']
    if params['optimizer'] == 'adam': opt = Adam(learning_rate=lr)
    elif params['optimizer'] == 'nadam': opt = Nadam(learning_rate=lr)
    elif params['optimizer'] == 'rmsprop': opt = RMSprop(learning_rate=lr)
    else: opt = Adam(learning_rate=lr)

    model.compile(optimizer=opt, loss='mse')
    return model

# ==========================================
# 4. KONFIGURACJA EKSPERYMENTÓW
# ==========================================

BASELINE_CONFIG = {
    'group': 'BASELINE',
    'num_layers': 1,
    'units': 64,
    'dense_neurons': 64,
    'optimizer': 'adam',
    'learning_rate': 0.001,
    'momentum': 0.0,
    'epochs': 50,
    'time_steps': 24
}

TEST_GROUPS = {
    'Layers':       {'param': 'num_layers',    'values': [1, 2, 3]},
    'Capacity':     {'param': 'units',         'values': [16, 32, 64, 128]},
    'Optimizer':    {'param': 'optimizer',     'values': ['adam', 'nadam', 'rmsprop']},
    'LearningRate': {'param': 'learning_rate', 'values': [0.01, 0.001, 0.0001]},
    'Epochs':       {'param': 'epochs',        'values': [20, 50, 100]}
}

experiments_queue = []
experiments_queue.append(BASELINE_CONFIG.copy())

for group_name, details in TEST_GROUPS.items():
    param_name = details['param']
    values = details['values']
    for val in values:
        new_cfg = BASELINE_CONFIG.copy()
        new_cfg['group'] = group_name

        if group_name == 'Capacity':
            new_cfg['units'] = val
            new_cfg['dense_neurons'] = val
        else:
            new_cfg[param_name] = val

        # Dodaj tylko jeśli różni się od Baseline (unikanie duplikatów logicznych)
        if new_cfg[param_name] != BASELINE_CONFIG.get(param_name) or group_name == 'Capacity':
             experiments_queue.append(new_cfg)

# Usuwanie duplikatów (jeśli jakieś zostały)
unique_experiments = []
seen_configs = set()
for exp in experiments_queue:
    config_tuple = tuple(sorted((k, v) for k, v in exp.items() if k != 'group'))
    if config_tuple not in seen_configs:
        seen_configs.add(config_tuple)
        unique_experiments.append(exp)

experiments_queue = unique_experiments
print(f"Zaplanowano {len(experiments_queue)} konfiguracji eksperymentalnych.")

Zaplanowano 12 konfiguracji eksperymentalnych.


In [ ]:
# ==========================================
# 5. PĘTLA EKSPERYMENTÓW
# ==========================================
N_REPEATS = 5
results = []
current_run = 0
total_runs = len(experiments_queue) * N_REPEATS

# --- Przygotowanie danych sekwencyjnych ---
ts = BASELINE_CONFIG['time_steps']
X_train, y_train_scaled = create_sequences(X_train_raw, y_train_raw, ts)
X_val, y_val_scaled = create_sequences(X_val_raw, y_val_raw, ts)
X_test, y_true_test = create_sequences(X_test_raw, y_test_raw_orig, ts)

# Prawdziwe wartości do metryk (odwrócone skalowanie)
y_true_train = scaler_y.inverse_transform(y_train_scaled)
y_true_val = scaler_y.inverse_transform(y_val_scaled)

for cfg in experiments_queue:
    # Ustalanie co badamy w tym kroku (do printowania)
    tested_param = "BASELINE"
    tested_val = "N/A"
    if cfg['group'] != 'BASELINE':
        diff = {k: cfg[k] for k in cfg if k in BASELINE_CONFIG and cfg[k] != BASELINE_CONFIG[k]}
        if diff:
            tested_param = list(diff.keys())[0]
            tested_val = diff[tested_param]
        if cfg['group'] == 'Capacity': tested_param = 'units'; tested_val = cfg['units']

    print(f"\n>>> BADANIE: {cfg['group']} | {tested_param} = {tested_val}")

    metrics = {k: [] for k in ['Train_R2', 'Val_R2', 'Test_R2',
                               'Train_MAE', 'Val_MAE', 'Test_MAE',
                               'Train_MSE', 'Val_MSE', 'Test_MSE']}

    for i in range(N_REPEATS):
        current_run += 1

        model = build_lstm_model((X_train.shape[1], X_train.shape[2]), cfg)

        # EarlyStopping dla wszystkich grup poza 'Epochs' (tam chcemy wymusić konkretną liczbę)
        cbs = [] if cfg['group'] == 'Epochs' else [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]

        model.fit(X_train, y_train_scaled, validation_data=(X_val, y_val_scaled),
                  epochs=cfg['epochs'], batch_size=256, verbose=0, callbacks=cbs)

        p_train = scaler_y.inverse_transform(model.predict(X_train, verbose=0))
        p_val   = scaler_y.inverse_transform(model.predict(X_val, verbose=0))
        p_test  = scaler_y.inverse_transform(model.predict(X_test, verbose=0))

        if np.isnan(p_test).any():
            print("! UWAGA: NaN w wynikach.")
            # Wypełniamy wartościami "karnymi"
            scores = [0]*3 + [999]*6
        else:
            scores = [
                r2_score(y_true_train, p_train), r2_score(y_true_val, p_val), r2_score(y_true_test, p_test),
                mean_absolute_error(y_true_train, p_train), mean_absolute_error(y_true_val, p_val), mean_absolute_error(y_true_test, p_test),
                mean_squared_error(y_true_train, p_train), mean_squared_error(y_true_val, p_val), mean_squared_error(y_true_test, p_test)
            ]

        for key, val in zip(metrics.keys(), scores):
            metrics[key].append(val)

        print(f"    Run {current_run}/{total_runs}: Test R2={metrics['Test_R2'][-1]:.4f} | MSE={metrics['Test_MSE'][-1]:.2f}")

    # Zapis średnich wyników dla konfiguracji
    res = cfg.copy()
    for key, val_list in metrics.items():
        res[f'{key}_Mean'] = np.mean(val_list)

    res['Tested_Value'] = tested_val
    results.append(res)

# Zapis do pliku
df_res = pd.DataFrame(results)
filename_out = 'wyniki_lstm_jena.csv'
df_res.to_csv(filename_out, index=False)
print(f"\nWyniki główne zapisano w: {filename_out}")


>>> BADANIE: BASELINE | BASELINE = N/A


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Wczytanie danych
FILENAME = 'wyniki_lstm_jena.csv'
df_res = pd.read_csv(FILENAME)

# ==========================================
# KONFIGURACJA STYLU (WEDŁUG SCHEMATU)
# ==========================================
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 14,          # Bazowy rozmiar
    'axes.titlesize': 20,     # Tytuły poszczególnych wykresów
    'xtick.labelsize': 18,    # Liczby na osi X
    'ytick.labelsize': 18,    # Liczby na osi Y
    'legend.fontsize': 14,    # Legenda
    'figure.titlesize': 24    # Tytuł główny
})

param_map = {
    'Layers': 'num_layers',
    'Capacity': 'units',
    'LearningRate': 'learning_rate',
    'Optimizer': 'optimizer',
    'Epochs': 'epochs'
}

baseline_row = df_res[df_res['group'] == 'BASELINE']
test_groups = [g for g in df_res['group'].unique() if g != 'BASELINE']

for group in test_groups:
    if group not in param_map: continue

    x_col = param_map[group]
    subset = df_res[df_res['group'] == group].copy()

    # Dodanie baseline jeśli istnieje
    if not baseline_row.empty:
        subset = pd.concat([subset, baseline_row], ignore_index=True)

    # ---------------------------------------------------------
    # SORTOWANIE (aby wykres szedł od najmniejszej do największej wartości)
    # ---------------------------------------------------------
    try:
        # Próbujemy sortować numerycznie (ważne dla Learning Rate, Epochs itp.)
        subset['sort_val'] = pd.to_numeric(subset[x_col], errors='coerce')

        subset = subset.sort_values('sort_val')
    except:
        # Fallback dla stringów (np. Optimizer)
        subset = subset.sort_values(x_col)

    # ---------------------------------------------------------
    # KLUCZ DO RÓWNYCH ODSTĘPÓW (MAPPING)
    # ---------------------------------------------------------
    # Tworzymy listę pozycji 0, 1, 2... dla każdego punktu
    x_positions = np.arange(len(subset))
    # Etykiety to rzeczywiste wartości z kolumny parametru
    x_labels = subset[x_col].astype(str).values

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'LSTM: Wpływ parametru {group}', fontsize=24, fontweight='bold')

    # Konfiguracja metryk (Nazwa wykresu, kolumna Train, kolumna Val, kolumna Test)
    # Dostosowałem nazwy kolumn do Twojego DataFrame (z sufiksami _Mean)
    metrics_config = [
        ('MAE', 'Train_MAE_Mean', 'Val_MAE_Mean', 'Test_MAE_Mean'),
        ('MSE', 'Train_MSE_Mean', 'Val_MSE_Mean', 'Test_MSE_Mean'),
        ('R2', 'Train_R2_Mean', 'Val_R2_Mean', 'Test_R2_Mean')
    ]

    for i, (name, tr_col, val_col, test_col) in enumerate(metrics_config):
        ax = axes[i]

        # Rysujemy względem pozycji (0, 1, 2...), a nie wartości
        # Styl linii wzięty ze schematu: Train(o), Val(s), Test(^, --)
        ax.plot(x_positions, subset[tr_col], marker='o', label='Train', linewidth=2)
        ax.plot(x_positions, subset[val_col], marker='s', label='Val', linewidth=2)
        ax.plot(x_positions, subset[test_col], marker='^', label='Test', linewidth=2, linestyle='--')

        ax.set_title(name)

        # Ustawiamy etykiety w równych odstępach
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)

        # Rotacja dla Optimizera, żeby napisy na siebie nie wchodziły
        if group == 'Optimizer':
            ax.tick_params(axis='x', rotation=45)

        ax.legend()
        ax.grid(True) # Opcjonalnie, whitegrid i tak to robi, ale dla pewności

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
import random

# ==========================================
# 6. RANDOM SEARCH
# ==========================================

N_RANDOM_SAMPLES = 30 # Liczba losowych kombinacji do przetestowania

N_REPEATS = 2

print(f"\n{'='*40}")
print(f"Rozpoczynam RANDOM SEARCH ({N_RANDOM_SAMPLES} prób)\n")

random_experiments_queue = []

# Zaktualizowana przestrzeń poszukiwań dla modelu LSTM
search_space = {
    'num_layers': [ 1,2],
    'units': [ 32, 64, 128,256],
    'dense_neurons': [ 32,64,128,256],
    'optimizer': ['adam', 'nadam'],
    'learning_rate': [0.01,0.001,0.0001],
    'epochs': [25, 50, 100],k
}


fixed_time_steps = 24 # Z oryginalnego BASELINE_CONFIG
fixed_momentum = 0.0
# --------------------------------------------------------------

for _ in range(N_RANDOM_SAMPLES):
    # Tworzymy nową konfigurację od podstaw, uwzględniając stałe parametry
    random_cfg = {
        'group': 'RANDOM_SEARCH',
        'time_steps': fixed_time_steps,
        'momentum': fixed_momentum,
    }

    # Losujemy wartości dla każdego parametru z przestrzeni poszukiwań
    for param, values_list in search_space.items():
        random_cfg[param] = random.choice(values_list)

    random_experiments_queue.append(random_cfg)

print(f"Wygenerowano {len(random_experiments_queue)} losowych konfiguracji do przetestowania.")

random_search_results = []
current_run_rs = 0
total_runs_rs = len(random_experiments_queue) * N_REPEATS

for cfg in random_experiments_queue:
    current_run_rs_exp = 0
    # Zaktualizowany string printujący konfigurację dla LSTM
    print(f"\n>>> RANDOM SEARCH: Konfiguracja {cfg.get('group', 'UNKNOWN')} | " \
          f"Layers: {cfg.get('num_layers', 'N/A')}, Units: {cfg.get('units', 'N/A')}, Dense: {cfg.get('dense_neurons', 'N/A')}, LR: {cfg.get('learning_rate', 'N/A')}")

    metrics = {k: [] for k in ['Train_R2', 'Val_R2', 'Test_R2',
                               'Train_MAE', 'Val_MAE', 'Test_MAE',
                               'Train_MSE', 'Val_MSE', 'Test_MSE']}

    for i in range(N_REPEATS):
        current_run_rs += 1
        current_run_rs_exp += 1


        model = build_lstm_model((X_train.shape[1], X_train.shape[2]), cfg)

        # EarlyStopping tylko jeśli nie testujemy 'Epochs' i jest dostępne w konfiguracji
        cbs = []
        if cfg.get('group') != 'Epochs':
             cbs.append(EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True))

        model.fit(X_train, y_train_scaled, validation_data=(X_val, y_val_scaled),
                  epochs=cfg['epochs'], batch_size=128, verbose=0, callbacks=cbs)

        p_train = scaler_y.inverse_transform(model.predict(X_train, verbose=0))
        p_val   = scaler_y.inverse_transform(model.predict(X_val, verbose=0))
        p_test  = scaler_y.inverse_transform(model.predict(X_test, verbose=0))

        if np.isnan(p_test).any():
            print("! UWAGA: NaN w wynikach. Przypisuję błędy domyślne.")
            scores = [0]*3 + [999]*6
        else:
            scores = [
                r2_score(y_true_train, p_train), r2_score(y_true_val, p_val), r2_score(y_true_test, p_test),
                mean_absolute_error(y_true_train, p_train), mean_absolute_error(y_true_val, p_val), mean_absolute_error(y_true_test, p_test),
                mean_squared_error(y_true_train, p_train), mean_squared_error(y_true_val, p_val), mean_squared_error(y_true_test, p_test)
            ]

        for key, val in zip(metrics.keys(), scores):
            metrics[key].append(val)

        print(f"    Run {current_run_rs}/{total_runs_rs} (Exp {current_run_rs_exp}/{N_REPEATS}): Test R2={metrics['Test_R2'][-1]:.4f} | MSE={metrics['Test_MSE'][-1]:.2f}")

    res = cfg.copy()
    for key, val_list in metrics.items():
        res[f'{key}_Mean'] = np.mean(val_list)

    random_search_results.append(res)

# ==========================================
# ZAPIS WYNIKÓW RANDOM SEARCH I WYBÓR NAJLEPSZEGO MODELU
# ==========================================
df_random_res = pd.DataFrame(random_search_results)
filename_random_out = 'wyniki_lstm_random_search.csv' # Zmieniona nazwa pliku
df_random_res.to_csv(filename_random_out, index=False)

print(f"\n{'='*40}")
print(f"             KONIEC RANDOM SEARCH        ")
print(f"{'='*40}")
print(f"Pełne wyniki random search zapisano w: {filename_random_out}")

# Znajdowanie najlepszego modelu
best_model_rs = df_random_res.loc[df_random_res['Test_R2_Mean'].idxmax()]

print("\n--- NAJLEPSZY MODEL Z RANDOM SEARCH (wg. Test_R2_Mean) ---")
print(f"Konfiguracja:")
# Zaktualizowana lista kluczy do wyświetlenia
for key in ['num_layers', 'units', 'dense_neurons', 'optimizer', 'learning_rate', 'epochs', 'time_steps']:
    if key in best_model_rs:
        print(f"  {key}: {best_model_rs[key]}")

print(f"Wyniki (średnie z {N_REPEATS} powtórzeń):")
print(f"  Train R2: {best_model_rs['Train_R2_Mean']:.4f}")
print(f"  Val   R2: {best_model_rs['Val_R2_Mean']:.4f}")
print(f"  Test  R2: {best_model_rs['Test_R2_Mean']:.4f}")
print(f"  Train MAE: {best_model_rs['Train_MAE_Mean']:.4f}")
print(f"  Val   MAE: {best_model_rs['Val_MAE_Mean']:.4f}")
print(f"  Test  MAE: {best_model_rs['Test_MAE_Mean']:.4f}")
print(f"  Train MSE: {best_model_rs['Train_MSE_Mean']:.4f}")
print(f"  Val   MSE: {best_model_rs['Val_MSE_Mean']:.4f}")
print(f"  Test  MSE: {best_model_rs['Test_MSE_Mean']:.4f}")


Rozpoczynam RANDOM SEARCH (30 prób)

Wygenerowano 30 losowych konfiguracji do przetestowania.

>>> RANDOM SEARCH: Konfiguracja RANDOM_SEARCH | Layers: 1, Units: 128, Dense: 128, LR: 0.0001


KeyboardInterrupt: 